# Deliverable 4: Apache Airflow DAG Orchestration & End-to-End Execution
**Course**: Modern Data Engineering for AI Systems Capstone (SDAIA Academy)

This notebook demonstrates the orchestrated pipeline execution wiring all 5 deliverables together:
1. **Stage 1 (Ingestion & DLQ)**: Pydantic Data Contract Validation & DLQ Routing.
2. **Stage 2 (Delta Lakehouse)**: Bronze Loading, Silver MERGE Upsert, Gold Aggregations.
3. **Stage 3 (Quality Gate)**: Great Expectations Suite (halting on failure).
4. **Stage 4 (Lineage)**: OpenLineage START, COMPLETE, FAIL lifecycle event tracking.
5. **Stage 5 (RAG Engine)**: Hybrid Search Indexing & Grounded Q&A Generation.

In [ ]:
import sys
import uuid
sys.path.append('..')

from src.ingestion.producer import publish_events
from src.ingestion.consumer import process_ingestion
from src.lakehouse.bronze_loader import load_bronze
from src.lakehouse.silver_merge import upsert_silver
from src.quality.ge_suite import run_silver_quality_gate
from src.lakehouse.gold_aggregator import build_gold_aggregates
from src.rag.rag_engine import GroundedRAGEngine
from src.quality.lineage_tracker import PipelineLineageTracker

## Execute Orchestrated Pipeline Sequential Run

In [ ]:
run_id = str(uuid.uuid4())
tracker = PipelineLineageTracker()

print("=== STAGE 1: Lineage Start ===")
tracker.emit_event('end_to_end_capstone_pipeline', 'START', run_id=run_id)

print("\n=== STAGE 2: Kafka Ingestion & DLQ Quarantine ===")
publish_events()
ingest_res = process_ingestion()

print("\n=== STAGE 3: Delta Lakehouse Bronze & Silver MERGE ===")
load_bronze()
upsert_silver()

print("\n=== STAGE 4: Great Expectations Quality Gate ===")
ge_res = run_silver_quality_gate()
if not ge_res['quality_gate_passed']:
    tracker.emit_event('end_to_end_capstone_pipeline', 'FAIL', run_id=run_id)
    raise ValueError('[PIPELINE HALT] Quality gate failed!')

print("\n=== STAGE 5: Delta Lakehouse Gold Aggregations ===")
build_gold_aggregates()

print("\n=== STAGE 6: Hybrid RAG Vector Index & Retrieval ===")
rag = GroundedRAGEngine()
rag.initialize_index()
answer_res = rag.query("What are the benefits of Hybrid RAG and Reciprocal Rank Fusion?")
print(answer_res['answer'])

print("\n=== STAGE 7: Lineage Complete ===")
tracker.emit_event('end_to_end_capstone_pipeline', 'COMPLETE', run_id=run_id)
print("\n[SUCCESS] End-to-End Orchestrated Pipeline Executed Cleanly!")